# PKG Contagion Phase 1 — inflow shock event study (v2)

**PNC Treasury Management · DSI · Payment Knowledge Graph (PKG)**

What happens to a TM organisation after its regular incoming payments **fall** (a payer stops,
skips or pays much less) or **rise** (an existing payer pays much more, or a new regular payer
starts)? Each event is compared with matched customers that had no event, 6 months before to
6 months after, on payment behaviour, deposits, revenue and credit.

**Changes from v1** (after the first real run, 2026-11-14)

- **Timing shifts cancel.** Lost dollars are counted over months e-1, e, e+1, so a payment that only moved
  across a month end is no longer a shock. A break needs a net shortfall of at least 50% of the expected amount.
- **Events start 2025-01**, when the 6 shock-free months before an event can be checked.
- **Headline = operating businesses, non-hub payers.** PNC's own entities, NAICS 52 and 92 and hub
  (processor) payers become separate cuts.
- **Panel sampled by stratum** (headline / other payer-wide / single-link, per direction), so cuts and dose
  response compare like with like.
- **Dollars and counts as average log change** (shown as %), instead of the median index that sat at 100 for counts.
- **Both directions.** Losses and gains are netted against each other in the same month, so a payer that
  changes its bank (old stream stops, new one starts) cancels out.
- **Deposits, revenue and credit** from `pkg_dep_cust_month` and `pkg_tm_cust_month`.
- **Pass-through:** % change in outflow per 1% change in inflow, by direction, size band and segment. This is
  the transmission rate a network propagation model needs.

| § | What it does | Output |
|---|---|---|
| 0 | Inputs: required columns, months, outcome families | one table |
| 1 | Population: TM organisations, standing streams, baseline inflow, segment, size band | one table |
| 2 | Stream breaks and gains, timing shifts, customer events | one table |
| 3 | Pre-flight counts, gates, headline choice | tables + f01, f02 |
| 4 | Matched controls and the event panel | one table |
| 5 | Effects, dose response, cuts, pass-through, pre-trend check | tables + f03-f11, f15, f17 |
| 6 | Second hop: do the customer's own regular payees feel it? | table + f14 |
| 7 | Case panels | table + f16 |
| 8 | Answer sheet, gates, files written | one table |

Tables are shown inline (pandas, one output per cell). **Figures are saved to disk, never shown inline.**
Nothing is written to cdi_tm. Work files go to the personal HDFS folder, results to `OUT_DIR`.

In [ ]:
# =============================================================================
# SETUP — environment shims, CONFIG, Spark session, helpers
# =============================================================================
import os, sys, time, json, math, hashlib, warnings, threading, contextlib
import numpy as np
import pandas as pd

# --- environment compat (must run before any toPandas / createDataFrame) ------
for _a, _t in [("object0", object), ("bool8", np.bool_), ("int0", np.intp),
               ("uint0", np.uintp), ("bool", np.bool_)]:
    if not hasattr(np, _a):
        setattr(np, _a, _t)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _w in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_w)

import matplotlib
matplotlib.use("Agg")                       # figures go to disk, never inline
import matplotlib.pyplot as plt
import matplotlib.ticker

from pyspark.sql import SparkSession, functions as F, Window as W

DB = "bdahd01p_dlcdi1_cdi_tm"

CONFIG = dict(
    # --- inputs ---------------------------------------------------------------
    MET_TABLE  = f"{DB}.pkg_cust_metrics_monthly_v3",
    PAIR_TABLE = f"{DB}.pkg_cust_pair_month_v3",
    CPTY_TABLE = f"{DB}.pkg_cpty_metrics_monthly",     # ent_id, month, hub_tier
    USE_HUB    = True,

    # --- outcome families outside the customer metrics (names from the §0 run of 2026-11-14)
    # kind: logpct = average log change, shown as %; diff = change from own baseline;
    #       scaled = change from own baseline / baseline monthly inflow, in pp.
    # subset_pos: measure only for customers whose baseline value of that output is above zero.
    # mask: drop a customer's window when `col` > 0 in one of `months` (acquisition months).
    EXTRA_OUTCOMES = [
        dict(name="deposits", table=f"{DB}.pkg_dep_cust_month", id_col="customer_pwr_id",
             id_kind="customer_pwr_id", month_col="month",
             mask=dict(col="n_acct_new", months=["2025-12", "2026-06"]),
             measures={
                 "dep_bal_mean":      dict(src="dep_bal_mean", kind="logpct", label="Average balance"),
                 "dep_bal_vs_inflow": dict(src="dep_bal_mean", kind="scaled",
                                           label="Average balance change (pp of monthly inflow)"),
                 "dep_bal_min":       dict(src="dep_bal_min", kind="scaled",
                                           label="Lowest balance change (pp of monthly inflow)"),
                 "dep_drawdown_pct":  dict(src="dep_drawdown_pct", kind="diff", label="Largest drawdown in month (change)"),
                 "days_negative":     dict(src="days_negative", kind="diff", label="Days with negative balance"),
                 "n_acct_closed":     dict(src="n_acct_closed", kind="diff", label="Accounts closed per month"),
             }),
        dict(name="revenue", table=f"{DB}.pkg_tm_cust_month", id_col="customer_pwr_id",
             id_kind="customer_pwr_id", month_col="month",
             measures={
                 "gross_rev_12m":      dict(src="gross_rev_12m", kind="logpct", label="Gross revenue, trailing 12m"),
                 "tm_rev_12m":         dict(src="tm_rev_12m", kind="logpct", label="TM revenue, trailing 12m"),
                 "credit_rev_12m":     dict(src="credit_rev_12m", kind="logpct", label="Credit revenue, trailing 12m"),
                 "non_credit_rev_12m": dict(src="non_credit_rev_12m", kind="logpct", label="Non-credit revenue, trailing 12m"),
             }),
        dict(name="credit", table=f"{DB}.pkg_tm_cust_month", id_col="customer_pwr_id",
             id_kind="customer_pwr_id", month_col="month", subset_pos="dhe",
             measures={
                 "dhe":           dict(src="dhe", kind="logpct", label="Direct hard exposure (DHE)"),
                 "pd_pct_now":    dict(src="pd_pct_now", kind="diff", label="PD (pd_pct_now, change)"),
                 "is_default":    dict(src="is_default", kind="diff", mult=100.0, label="In default (pp)"),
                 "past_due_days": dict(src="past_due_days", kind="diff", label="Days past due"),
             }),
    ],

    # --- window ---------------------------------------------------------------
    MONTH_MIN = "2024-01",
    MONTH_MAX = None,            # None = last month present in both inputs

    # --- run control ----------------------------------------------------------
    RUN_STAGE = "full",          # "preflight" stops after §3
    REBUILD   = False,           # True = ignore the work files of this configuration
    WORK_DIR  = "hdfs:///user/pk36814/pkg_contagion_p1/work",
    OUT_DIR   = "../metrics/contagion/p1",
    HDFS_REPLICATION   = 2,
    SHUFFLE_PARTITIONS = 800,

    # --- population -----------------------------------------------------------
    ORG_PARTY_TYPES = ["O"],
    BASE_MONTHS     = 6,         # baseline = the 6 months before the event month
    BASE_IN_MIN     = 10_000.0,  # minimum average monthly incoming dollars at baseline
    SIZE_BANDS      = [100_000.0, 1_000_000.0],   # baseline monthly inflow: small / mid / large
    FIN_PUBLIC_NAICS2 = ["52", "92"],
    PNC_NAME_REGEX  = r"\bPNC\b",

    # --- stream events (monthly grain) ----------------------------------------
    STANDING_MIN = 4,            # paid in >= 4 of the last 6 months, incl. the last one
    LOSS_MIN     = 0.50,         # net shortfall over e-1..e+1 >= 50% of the expected amount = break
    GAIN_MIN     = 0.50,         # paid >= 150% of expected in both e and e+1 = growth
    NEW_MONTHS   = 3,            # new payer: paid in e, e+1, e+2 and not in the 6 months before

    # --- customer events ------------------------------------------------------
    TIER_EDGES    = [0.10, 0.20, 0.50],   # net event size / baseline inflow; first edge = minimum kept
    HEADLINE_MIN  = 0.20,
    CLEAN_MONTHS  = 6,           # event = first event after 6 months without one (either direction)
    MIN_FOLLOW    = 3,           # event months end 3 months before the last month
    PAYERWIDE_SHARE      = 0.50,
    PAYERWIDE_MIN_PAYEES = 5,
    HUB_TIER      = 2,

    # --- gates ------------------------------------------------------------------
    GATE_MIN_EVENTS    = 1000,
    GATE_MIN_PAYERWIDE = 0.10,

    # --- event study ----------------------------------------------------------
    PRE_MONTHS  = 6,
    POST_MONTHS = 6,
    N_CONTROLS  = 3,
    CONTROL_CLEAN_AFTER = 0,     # controls: no event in the 6 months before nor in the event month
    PANEL_PER_STRATUM   = 5000,  # events per stratum (direction x headline / payer-wide / single-link)
    EFFECT_T    = 3,
    BOOT_REPS   = 200,
    LOG_CAP     = 10.0,          # log changes capped at x10 / /10
    SEED        = 20261114,

    # --- cases ----------------------------------------------------------------
    CASE_TOP_LOSS = 6,
    CASE_TOP_GAIN = 4,
    CASE_NAME_PATTERNS = ["UNITED CONSUMER FINANCIAL"],
)

DIRS = ["loss", "gain"]
LOSS_TYPES = ["stop", "skip", "short", "open"]
GAIN_TYPES = ["existing payer grew", "new payer"]

# Outcomes from the customer metrics: (label, kind, multiplier). Core ones are required;
# optional ones are dropped with a note if the table does not have them.
OUTCOMES = {
    "amt_all_in":               ("Incoming dollars",                          "logpct", 1.0),
    "amt_all_out":              ("Outgoing dollars",                          "logpct", 1.0),
    "ntxn_all_out":             ("Outgoing payment count",                    "logpct", 1.0),
    "active_days_out":          ("Days with outgoing payments",               "logpct", 1.0),
    "ncpty_ext_out":            ("External payees paid",                      "logpct", 1.0),
    "rec_standing_out":         ("Standing payees",                           "logpct", 1.0),
    "rec_broken_share_out":     ("Standing payees not paid (pp)",             "diff",   100.0),
    "rec_broken_amt_share_out": ("Standing payee dollars not paid (pp)",      "diff",   100.0),
    "net_flow_all":             ("Net flow (pp of monthly inflow)",           "scaled", 1.0),
}
OPTIONAL_OUTCOMES = {
    "new_cpty_ext_out":         ("New external payees per month",             "diff",   1.0),
    "lost_cpty_ext_out":        ("External payees dropped per month",         "diff",   1.0),
}
MET_REQUIRED  = ["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type",
                 "naics2", "is_tm", "pwr_shared", "month", "rec_standing_in"] + list(OUTCOMES)
PAIR_REQUIRED = ["cust", "dir", "scope", "other", "amount", "month"]
CPTY_REQUIRED = ["ent_id", "month", "hub_tier"]

# --- Spark session (ACC code-review limits: 4g / 4g / 1g, 512 executors) --------
spark = (SparkSession.builder.appName("pkg_contagion_p1")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g")
         .config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g")
         .config("spark.executor.memoryOverhead", "1g")
         .config("spark.yarn.am.memory", "4g")
         .config("spark.yarn.am.memoryOverhead", "1g")
         .config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.kryoserializer.buffer.max", "128m")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))
         .config("spark.hadoop.dfs.replication", str(CONFIG["HDFS_REPLICATION"]))
         .enableHiveSupport().getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:    # cluster noise: Atlas lineage hook, Kafka, netty (harmless to data)
    _L = spark._jvm.org.apache.logging.log4j
    for _n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage",
               "org.apache.kafka", "org.apache.spark.sql.catalyst.util.package",
               "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    pass
spark.conf.set("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))

# --- display: one HTML output per cell; progress lines above the tables --------
try:
    from IPython.display import display, HTML
    _NB = "ipykernel" in sys.modules
except Exception:
    _NB = False


def _fmt(v):
    if v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))):
        return ""
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        a = abs(float(v))
        if a >= 1000:
            return f"{v:,.0f}"
        if a >= 10:
            return f"{v:,.1f}"
        return f"{v:,.3f}".rstrip("0").rstrip(".") if a else "0"
    return str(v)


def _html(df):
    d = df.copy()
    for c in d.columns:
        d[c] = d[c].map(_fmt)
    return d.to_html(index=False, border=0, escape=True)


class _Cell:
    """Progress lines and result tables of one cell, rendered as a single output."""
    def __init__(self):
        self.lines, self.tables, self.h = [], [], None

    def start(self, title):
        self.title, self.lines, self.tables = title, [], []
        self.h = display(HTML(""), display_id=True) if _NB else None
        self._draw()

    def log(self, txt, replace_last=False):
        if replace_last and self.lines:
            self.lines[-1] = txt
        else:
            self.lines.append(txt)
        self._draw()
        if not _NB:
            print("   ", txt, flush=True)

    def show(self, *blocks):
        self.tables = list(blocks)
        self._draw()
        if not _NB:
            for lab, df in blocks:
                print(f"\n== {lab}")
                print(df.to_string(index=False, max_rows=80))

    def _draw(self):
        if self.h is None:
            return
        css = ("<style>.pkgc table{border-collapse:collapse;font-size:12px;margin:2px 0 12px}"
               ".pkgc th{background:#f1f0ec;color:#0b0b0b;text-align:left;padding:3px 8px;border-bottom:1px solid #b8b7b0}"
               ".pkgc td{padding:2px 8px;border-bottom:1px solid #e6e5e0}"
               ".pkgc h4{margin:10px 0 3px;font-size:13px}.pkgc .pg{color:#8a8985;font-size:12px}</style>")
        body = f"<div class='pkgc'><h3 style='margin:4px 0'>{self.title}</h3>"
        body += "".join(f"<div class='pg'>{x}</div>" for x in self.lines)
        body += "".join(f"<h4>{lab}</h4>{_html(df)}" for lab, df in self.tables)
        self.h.update(HTML(css + body + "</div>"))


CELL = _Cell()
TIMINGS = []


@contextlib.contextmanager
def step(name):
    """Timed step with a live line: elapsed minutes and Spark tasks of the running stage."""
    t0 = time.time()
    CELL.log(f"running: {name} (started {time.strftime('%H:%M')})")
    stop = threading.Event()

    def _tick():
        while not stop.wait(30):
            msg = f"running: {name} ({(time.time() - t0) / 60:.1f} min"
            try:
                st = spark.sparkContext.statusTracker()
                ids = st.getActiveStageIds()
                if ids:
                    info = st.getStageInfo(max(ids))
                    msg += f", stage {info.stageId}: {info.numCompletedTasks:,}/{info.numTasks:,} tasks"
            except Exception:
                pass
            if _NB:
                CELL.log(msg + ")", replace_last=True)

    th = threading.Thread(target=_tick, daemon=True)
    th.start()
    try:
        yield
    finally:
        stop.set()
        mins = (time.time() - t0) / 60
        TIMINGS.append((name, round(mins, 2)))
        if _NB:
            CELL.log(f"done: {name} ({mins:.1f} min)", replace_last=True)
        else:
            print(f"    done: {name} ({mins:.1f} min)", flush=True)


GATES = []


def gate(name, ok, detail, action=""):
    GATES.append(dict(gate=name, result="PASS" if ok else "FAIL", detail=detail, action=action))


def gates_frame():
    return pd.DataFrame(GATES, columns=["gate", "result", "detail", "action"])


# --- months ---------------------------------------------------------------------
_BASE = int(CONFIG["MONTH_MIN"][:4]) * 12 + int(CONFIG["MONTH_MIN"][5:7])


def mi_col(c):
    """Integer month index, 0 = MONTH_MIN. Gaps are safe with rangeBetween windows."""
    c = F.col(c) if isinstance(c, str) else c
    return (F.substring(c, 1, 4).cast("int") * 12 + F.substring(c, 6, 2).cast("int") - F.lit(_BASE)).cast("int")


def mi2m(i):
    k = int(i) + _BASE
    return f"{(k - 1) // 12:04d}-{(k - 1) % 12 + 1:02d}"


def m2mi(m):
    return int(m[:4]) * 12 + int(m[5:7]) - _BASE


def table_months(tbl):
    try:
        p = spark.sql(f"SHOW PARTITIONS {tbl}").toPandas().iloc[:, 0]
        m = sorted({x.split("month=")[1].split("/")[0] for x in p if "month=" in x})
        if m:
            return m
    except Exception:
        pass
    return sorted(r[0] for r in spark.table(tbl).select("month").distinct().collect())


# --- work files: one folder per configuration, so a changed threshold never reuses old files
_SIG_KEYS = ["MET_TABLE", "PAIR_TABLE", "CPTY_TABLE", "USE_HUB", "MONTH_MIN", "ORG_PARTY_TYPES",
             "BASE_MONTHS", "BASE_IN_MIN", "SIZE_BANDS", "FIN_PUBLIC_NAICS2", "PNC_NAME_REGEX",
             "STANDING_MIN", "LOSS_MIN", "GAIN_MIN", "NEW_MONTHS", "TIER_EDGES", "CLEAN_MONTHS",
             "PAYERWIDE_SHARE", "PAYERWIDE_MIN_PAYEES", "HUB_TIER"]


def _work(name):
    return f"{CONFIG['WORK_DIR']}/v2_sig={SIG}/{name}"


def cache_parquet(sdf, name):
    """Write once, read back. Later runs with the same configuration skip the build."""
    path = _work(name)
    if not CONFIG["REBUILD"]:
        try:
            out = spark.read.parquet(path)
            out.limit(1).count()
            CELL.log(f"reused work file: {name}")
            return out
        except Exception:
            pass
    sdf.write.mode("overwrite").parquet(path)
    return spark.read.parquet(path)


# --- figures: saved to disk, indexed, never shown inline --------------------------
OUT_DIR = CONFIG["OUT_DIR"]
FIG_DIR = os.path.join(OUT_DIR, "figs")
os.makedirs(FIG_DIR, exist_ok=True)
for _f in os.listdir(FIG_DIR):                       # figures of an earlier run
    if _f[:1] == "f" and _f.endswith((".png", ".html")):
        os.remove(os.path.join(FIG_DIR, _f))
FIG_INDEX = []

INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb"
C_EVENT, C_CTRL = "#2a78d6", "#8a8985"          # event customers = blue, matched controls = gray
C_DIR = {"loss": "#2a78d6", "gain": "#eb6834"}  # direction (categorical slots 1, 2)
C_IN, C_OUT = "#008300", "#e34948"              # inflow green, outflow red (team chart rule)
TYPE_COLORS = {"stop": "#2a78d6", "skip": "#eb6834", "short": "#1baf7a", "open": "#eda100",
               "existing payer grew": "#2a78d6", "new payer": "#eb6834"}

plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": "#b8b7b0", "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "lines.linewidth": 2.0, "figure.dpi": 110,
})


def savefig(fig, name, section, caption):
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    FIG_INDEX.append(dict(file=f"{name}.png", section=section, caption=caption))
    return path


def save_plotly(fig, name, section, caption):
    """Standalone HTML copy (opens in the browser; inline Plotly is empty in this JupyterLab)."""
    try:
        fig.write_html(os.path.join(FIG_DIR, f"{name}.html"), include_plotlyjs=True, full_html=True)
        FIG_INDEX.append(dict(file=f"{name}.html", section=section, caption=caption + " (interactive)"))
    except Exception as ex:
        CELL.log(f"interactive copy of {name} skipped: {type(ex).__name__}")


def usd(v):
    return "" if v is None or (isinstance(v, float) and math.isnan(v)) else f"${v:,.0f}"


def ci(e, lo, hi):
    """'-9.6 (-10.4 to -8.9)'; '(n.s.)' when the interval includes zero."""
    if e is None or not np.isfinite(e):
        return ""
    f = (lambda v: f"{v:+.3f}") if max(abs(e), abs(lo if np.isfinite(lo) else 0), abs(hi if np.isfinite(hi) else 0)) < 0.1 \
        else (lambda v: f"{v:+.1f}")
    s = f"{f(e)} ({f(lo)} to {f(hi)})" if np.isfinite(lo) and np.isfinite(hi) else f(e)
    return s + ("" if (np.isfinite(lo) and np.isfinite(hi) and (lo > 0 or hi < 0)) else " n.s.")


ANSWERS = []


def ans(q, a):
    ANSWERS.append(dict(question=q, answer=a))

## §0 · Inputs

Confirms every column this notebook depends on exists and lists the months of each input. Outcome families
whose configured names are not in their table are listed and skipped, nothing else stops. A missing
required column stops the run here, before any heavy work.

In [ ]:
CELL.start("§0 · Inputs")
with step("schemas and months"):
    _req = [("customer metrics", CONFIG["MET_TABLE"], MET_REQUIRED),
            ("customer x counterparty pairs", CONFIG["PAIR_TABLE"], PAIR_REQUIRED)]
    if CONFIG["USE_HUB"]:
        _req.append(("counterparty metrics", CONFIG["CPTY_TABLE"], CPTY_REQUIRED))
    _rows, _missing, _months = [], {}, {}
    for _lab, _t, _need in _req:
        _cols = spark.table(_t).columns
        _miss = [c for c in _need if c not in _cols]
        _months[_t] = table_months(_t)
        _rows.append(dict(input=_lab, table=_t, columns=len(_cols),
                          months=f"{_months[_t][0]} .. {_months[_t][-1]} ({len(_months[_t])})",
                          missing_columns=", ".join(_miss) if _miss else "none"))
        if _miss:
            _missing[_t] = _miss
    _met_cols = set(spark.table(CONFIG["MET_TABLE"]).columns)
    for _c, _v in OPTIONAL_OUTCOMES.items():
        if _c in _met_cols:
            OUTCOMES[_c] = _v
    _opt_missing = [c for c in OPTIONAL_OUTCOMES if c not in _met_cols]
    _inputs = pd.DataFrame(_rows)

    _xrows, EXTRA_ACTIVE = [], []
    for _x in CONFIG["EXTRA_OUTCOMES"]:
        try:
            _names = spark.table(_x["table"]).columns
        except Exception:
            _xrows.append(dict(family=_x["name"], table=_x["table"], status="table not found", measures=""))
            continue
        _want = [_x["id_col"], _x["month_col"]] + [m["src"] for m in _x["measures"].values()]
        if _x.get("mask"):
            _want.append(_x["mask"]["col"])
        _miss = [w for w in _want if w not in _names]
        if _miss:
            _st = "skipped, not in table: " + ", ".join(_miss)
        else:
            _st = "active"
            EXTRA_ACTIVE.append(_x)
        try:
            _xm = table_months(_x["table"]) if not _miss else []
            _mtxt = f"{_xm[0]} .. {_xm[-1]} ({len(_xm)})" if _xm else ""
        except Exception:
            _mtxt = ""
        _xrows.append(dict(family=_x["name"], table=_x["table"], status=_st, months=_mtxt,
                           measures=", ".join(m["src"] for m in _x["measures"].values())))
    _extra = pd.DataFrame(_xrows)

CELL.show(("Required inputs", _inputs),
          ("Outcome families", _extra),
          ("Optional payment outcomes not in the metrics table", pd.DataFrame({"column": _opt_missing or ["none"]})))
if _missing:
    raise ValueError("Required columns not found, nothing was built: "
                     + "; ".join(f"{t}: {', '.join(m)}" for t, m in _missing.items()))

MONTH_MIN = CONFIG["MONTH_MIN"]
_common = sorted(set(_months[CONFIG["MET_TABLE"]]) & set(_months[CONFIG["PAIR_TABLE"]]))
_common = [m for m in _common if m >= MONTH_MIN and (CONFIG["MONTH_MAX"] is None or m <= CONFIG["MONTH_MAX"])]
BM, CLEAN = CONFIG["BASE_MONTHS"], CONFIG["CLEAN_MONTHS"]
if len(_common) < BM + CLEAN + CONFIG["MIN_FOLLOW"] + 3:
    raise ValueError(f"Only {len(_common)} common months between metrics and pairs: too few for the event study.")
MONTH_MAX = _common[-1]
MAX_MI = m2mi(MONTH_MAX)
EV_START, EV_END = BM + CLEAN, MAX_MI - CONFIG["MIN_FOLLOW"]
gate("months are continuous", len(_common) == MAX_MI + 1,
     f"{len(_common)} months present, {MAX_MI + 1} expected between {MONTH_MIN} and {MONTH_MAX}",
     "a missing month would look like a wave of stops")
SIG = hashlib.md5(json.dumps({k: CONFIG[k] for k in _SIG_KEYS} | {"max": MONTH_MAX},
                             sort_keys=True, default=str).encode()).hexdigest()[:10]
ans("Window", f"{MONTH_MIN} .. {MONTH_MAX}; events {mi2m(EV_START)} .. {mi2m(EV_END)}; configuration {SIG}")

## §1 · Population

TM organisations (`is_tm`, `party_type` O, power id not shared), one row per customer per month.
**Baseline inflow** = average monthly `amt_all_in` over the 6 months before. A customer-month is **eligible**
with at least one standing incoming relationship (`rec_standing_in`) and baseline inflow above the floor.

- **Segment:** PNC own (name matches PNC), finance & public (NAICS2 52 or 92), operating (everything else).
- **Size band:** baseline monthly inflow under 100K, 100K to 1M, 1M and over.

In [ ]:
CELL.start("§1 · Population")
with step("TM organisations, baseline inflow, eligibility"):
    _met = spark.table(CONFIG["MET_TABLE"]).where(F.col("month").between(MONTH_MIN, MONTH_MAX))
    _tm = _met.where((F.col("is_tm").cast("boolean") == F.lit(True))
                     & F.col("party_type").isin(CONFIG["ORG_PARTY_TYPES"])
                     & (F.coalesce(F.col("pwr_shared").cast("boolean"), F.lit(False)) == F.lit(False)))
    _pop = _tm.select(
        F.col("mdm_id").cast("string").alias("cust"),
        F.col("customer_pwr_id").cast("string").alias("customer_pwr_id"),
        F.coalesce(F.col("master_pwr_id").cast("string"), F.col("mdm_id").cast("string")).alias("master"),
        F.col("customer_name"),
        F.coalesce(F.col("naics2").cast("string"), F.lit("NA")).alias("naics2n"),
        mi_col("month").alias("mi"),
        F.col("rec_standing_in").cast("double").alias("rec_standing_in"),
        *[F.col(c).cast("double").alias(c) for c in OUTCOMES])
    _w6 = W.partitionBy("cust").orderBy("mi").rangeBetween(-BM, -1)
    _pop = _pop.withColumn("base_in", F.sum("amt_all_in").over(_w6) / F.lit(float(BM)))
    pop = cache_parquet(_pop, "pop")

    _b1, _b2 = CONFIG["SIZE_BANDS"]
    _elig = (pop.where((F.col("mi") >= BM) & (F.col("rec_standing_in") >= 1)
                       & (F.col("base_in") >= CONFIG["BASE_IN_MIN"]))
             .withColumn("pbin", F.ntile(100).over(W.partitionBy("mi").orderBy("base_in")))
             .withColumn("segment",
                         F.when(F.upper(F.coalesce(F.col("customer_name"), F.lit(""))).rlike(CONFIG["PNC_NAME_REGEX"]),
                                F.lit("PNC own"))
                         .when(F.col("naics2n").isin(CONFIG["FIN_PUBLIC_NAICS2"]), F.lit("finance & public"))
                         .otherwise(F.lit("operating")))
             .withColumn("size_band", F.when(F.col("base_in") < _b1, F.lit("small"))
                         .when(F.col("base_in") < _b2, F.lit("mid")).otherwise(F.lit("large")))
             .select("cust", "customer_pwr_id", "master", "customer_name", "naics2n", "mi", "base_in", "pbin",
                     "segment", "size_band"))
    elig = cache_parquet(_elig, "elig")

    _pm = (pop.groupBy("mi").agg(
        F.count(F.lit(1)).alias("tm_orgs"),
        F.sum((F.col("rec_standing_in") >= 1).cast("long")).alias("with_standing_in"),
        F.sum("amt_all_in").alias("in_usd"),
        F.sum(F.when(F.col("rec_standing_in") >= 1, F.col("amt_all_in")).otherwise(0.0)).alias("in_usd_standing"))
        .join(elig.groupBy("mi").agg(F.count(F.lit(1)).alias("eligible")), "mi", "left")
        .orderBy("mi").toPandas())
    _seg = (elig.where(F.col("mi") == MAX_MI).groupBy("segment", "size_band").count().toPandas()
            .pivot(index="segment", columns="size_band", values="count").fillna(0).astype(int)
            .reindex(columns=["small", "mid", "large"]).fillna(0).astype(int).reset_index())

if _pm["tm_orgs"].sum() == 0:
    raise ValueError("No TM organisation rows found. Check is_tm and ORG_PARTY_TYPES.")
_pm.insert(0, "month", _pm["mi"].map(mi2m))
_pm["eligible"] = _pm["eligible"].fillna(0).astype(int)
_pm["standing_share"] = _pm["with_standing_in"] / _pm["tm_orgs"]
_pm["standing_usd_share"] = _pm["in_usd_standing"] / _pm["in_usd"]
_last = _pm.iloc[-1]
CELL.show(("Population, last month", pd.DataFrame([dict(
              tm_orgs=int(_last.tm_orgs), with_standing_in=int(_last.with_standing_in),
              standing_share=float(_last.standing_share), standing_usd_share=float(_last.standing_usd_share),
              eligible=int(_last.eligible))])),
          ("Eligible customers, last month, by segment and size band", _seg),
          ("By month", _pm[["month", "tm_orgs", "with_standing_in", "standing_share", "standing_usd_share", "eligible"]]))
ans("TM organisations with a standing incoming relationship (last month)",
    f"{int(_last.with_standing_in):,} of {int(_last.tm_orgs):,} ({_last.standing_share:.1%}), "
    f"holding {_last.standing_usd_share:.1%} of incoming dollars")

## §2 · Stream breaks and gains, customer events

A **stream** is customer x payer (internal or external), incoming side of the pair table. A stream is
**standing** going into month e when it was paid in at least 4 of the 6 months before, including month e-1.
Expected amount = median of the last 3 paid months.

| Stream event | Rule |
|---|---|
| break | net shortfall over e-1, e, e+1 against 3 x expected is at least 50% of expected (shift-proof) |
| - stop | break, not paid in e nor e+1 |
| - skip | break, not paid in e, paid in e+1 |
| - short | break, paid in e but less |
| - open | break in the last month of data (cannot be typed yet) |
| grow | paid at least 150% of expected in both e and e+1 |
| new payer | paid in e, e+1, e+2, not paid in the 6 months before |

**Customer event** (per month e): net loss = lost dollars - grown dollars - new payers starting in e or e+1;
net gain = grown dollars + new payers starting in e - lost dollars. An event needs at least 10% of baseline
inflow. The first event after 6 event-free months is used, one per customer and direction, from 2025-01.
**Payer-wide** = the payer broke (or grew) at least half of its standing PNC payees that month, 5 payees or more.

In [ ]:
CELL.start("§2 · Stream breaks and gains, customer events")
SM, LOSS_MIN, GAIN_MIN = CONFIG["STANDING_MIN"], CONFIG["LOSS_MIN"], CONFIG["GAIN_MIN"]
TIER0 = CONFIG["TIER_EDGES"][0]
PWS, PWN = CONFIG["PAYERWIDE_SHARE"], CONFIG["PAYERWIDE_MIN_PAYEES"]

with step("stream windows (pair table, incoming side)"):
    _pair = (spark.table(CONFIG["PAIR_TABLE"])
             .where(F.col("month").between(MONTH_MIN, MONTH_MAX)
                    & (F.col("dir") == "in") & F.col("scope").isin("int", "ext"))
             .select(F.col("cust").cast("string").alias("cust"), "scope",
                     F.col("other").cast("string").alias("other"),
                     mi_col("month").alias("mi"), F.col("amount").cast("double").alias("amount"))
             .join(pop.select("cust").distinct(), "cust", "left_semi")
             .groupBy("cust", "scope", "other", "mi").agg(F.sum("amount").alias("amount"))
             .where(F.col("amount") > 0))
    _w = W.partitionBy("cust", "scope", "other").orderBy("mi")
    _win = (_pair
            .withColumn("cnt6", F.count(F.lit(1)).over(_w.rangeBetween(-5, 0)))
            .withColumn("prev_mi", F.lag("mi").over(_w))
            .withColumn("l1_mi", F.lead("mi").over(_w)).withColumn("l1_amt", F.lead("amount").over(_w))
            .withColumn("l2_mi", F.lead("mi", 2).over(_w)).withColumn("l2_amt", F.lead("amount", 2).over(_w))
            .withColumn("last3", F.collect_list("amount").over(_w.rowsBetween(-2, 0))))
    win = cache_parquet(_win, "stream_win")

with step("standing streams: breaks (net of timing shifts) and growth"):
    _z = F.lit(0.0)
    _s = (win.where((F.col("cnt6") >= SM) & (F.col("mi") < MAX_MI))
          .withColumn("e", F.col("mi") + 1)
          .withColumn("expected", F.expr("element_at(array_sort(last3), cast(ceil(size(last3) / 2.0) as int))"))
          .withColumn("a_e", F.when(F.col("l1_mi") == F.col("e"), F.col("l1_amt")).otherwise(_z))
          .withColumn("a_e1", F.when(F.col("l1_mi") == F.col("e") + 1, F.col("l1_amt"))
                      .when(F.col("l2_mi") == F.col("e") + 1, F.col("l2_amt")).otherwise(_z))
          .withColumn("has_e1", F.col("e") + 1 <= F.lit(MAX_MI))
          .withColumn("gross", F.greatest(F.col("expected") - F.col("a_e"), _z))
          .withColumn("net", F.when(F.col("has_e1"), F.lit(3.0)).otherwise(F.lit(2.0)) * F.col("expected")
                      - F.col("amount") - F.col("a_e") - F.col("a_e1"))
          .withColumn("lost_raw", F.greatest(F.least(F.col("gross"), F.col("net")), _z))
          .withColumn("gross_break", F.col("gross") >= LOSS_MIN * F.col("expected"))
          .withColumn("broke", F.col("lost_raw") >= LOSS_MIN * F.col("expected"))
          .withColumn("ev", F.when(~F.col("broke"), F.lit(None).cast("string"))
                      .when(F.col("a_e") > 0, F.lit("short"))
                      .when(~F.col("has_e1"), F.lit("open"))
                      .when(F.col("a_e1") == 0, F.lit("stop"))
                      .otherwise(F.lit("skip")))
          .withColumn("lost", F.when(F.col("broke"), F.col("lost_raw")).otherwise(_z))
          .withColumn("g_raw", F.when(F.col("has_e1"), F.least(F.col("a_e"), F.col("a_e1")) - F.col("expected")).otherwise(_z))
          .withColumn("grew", F.col("g_raw") >= GAIN_MIN * F.col("expected"))
          .withColumn("gain", F.when(F.col("grew"), F.col("g_raw")).otherwise(_z))
          .select("cust", "scope", "other", "e", "expected", F.col("amount").alias("a_prev"), "a_e", "a_e1",
                  "gross_break", "broke", "ev", "lost", "grew", "gain"))
    risk = cache_parquet(_s, "stream_risk")

with step("new regular payers"):
    _nm = CONFIG["NEW_MONTHS"]
    _new = (win.where((F.col("mi") >= BM) & (F.col("mi") + _nm - 1 <= MAX_MI)
                      & (F.col("prev_mi").isNull() | (F.col("mi") - F.col("prev_mi") > BM))
                      & (F.col("l1_mi") == F.col("mi") + 1)
                      & ((F.col("l2_mi") == F.col("mi") + 2) if _nm >= 3 else F.lit(True)))
            .select("cust", "scope", "other", F.col("mi").alias("e"),
                    (F.least(F.col("amount"), F.col("l1_amt"), F.col("l2_amt")) if _nm >= 3
                     else F.least(F.col("amount"), F.col("l1_amt"))).alias("new_amt")))
    new_raw = cache_parquet(_new, "new_raw")

with step("payer-month roll-up, payer-wide and hub flags"):
    _pr = risk.groupBy("scope", "other", "e").agg(
        F.count(F.lit(1)).alias("n_risk"),
        F.sum(F.col("broke").cast("int")).alias("n_broken"),
        F.sum(F.col("grew").cast("int")).alias("n_grow"))
    _pn = new_raw.groupBy("scope", "other", "e").agg(F.count(F.lit(1)).alias("n_new"))
    _pmo = (_pr.join(_pn, ["scope", "other", "e"], "full").na.fill(0, ["n_risk", "n_broken", "n_grow", "n_new"])
            .withColumn("pw_loss", ((F.col("n_risk") >= PWN) & (F.col("n_broken") >= PWS * F.col("n_risk"))).cast("int"))
            .withColumn("pw_gain", ((F.col("n_risk") >= PWN) & (F.col("n_grow") >= PWS * F.col("n_risk"))).cast("int"))
            .withColumn("pw_new", ((F.col("n_new") >= PWN)
                                   & (F.col("n_new") >= PWS * (F.col("n_risk") + F.col("n_new")))).cast("int")))
    if CONFIG["USE_HUB"]:
        _hub = (spark.table(CONFIG["CPTY_TABLE"])
                .where(F.col("month").between(MONTH_MIN, MONTH_MAX)
                       & (F.col("hub_tier").cast("int") >= CONFIG["HUB_TIER"]))
                .select(F.lit("ext").alias("scope"), F.col("ent_id").cast("string").alias("other"),
                        (mi_col("month") + 1).alias("e")).distinct().withColumn("is_hub", F.lit(1)))
        _pmo = _pmo.join(F.broadcast(_hub), ["scope", "other", "e"], "left")
    else:
        _pmo = _pmo.withColumn("is_hub", F.lit(None).cast("int"))
    payer_month = cache_parquet(_pmo.withColumn("is_hub", F.coalesce(F.col("is_hub"), F.lit(0))), "payer_month")

with step("stream events with flags"):
    _flags = payer_month.select("scope", "other", "e", "pw_loss", "pw_gain", "pw_new", "is_hub", "n_risk", "n_broken", "n_grow", "n_new")
    stream_events = cache_parquet(risk.where(F.col("broke") | F.col("grew")).join(_flags, ["scope", "other", "e"], "left"),
                                  "stream_events")
    new_streams = cache_parquet(new_raw.join(_flags, ["scope", "other", "e"], "left"), "new_streams")

with step("customer events (net of losses and gains)"):
    def _s_if(cond, col):
        return F.sum(F.when(cond, F.col(col)).otherwise(0.0))
    _cr = stream_events.groupBy("cust", "e").agg(
        F.sum("lost").alias("lost"),
        *[_s_if(F.col("ev") == t, "lost").alias(f"lost_{t}") for t in LOSS_TYPES],
        _s_if(F.col("scope") == "int", "lost").alias("lost_int"),
        _s_if(F.col("pw_loss") == 1, "lost").alias("lost_pw"),
        _s_if(F.col("is_hub") == 1, "lost").alias("lost_hub"),
        F.sum("gain").alias("grow"),
        _s_if(F.col("scope") == "int", "gain").alias("grow_int"),
        _s_if(F.col("pw_gain") == 1, "gain").alias("grow_pw"),
        _s_if(F.col("is_hub") == 1, "gain").alias("grow_hub"),
        F.sum(F.col("broke").cast("int")).alias("n_broken"),
        F.sum(F.col("grew").cast("int")).alias("n_grew"))
    _cn = new_streams.groupBy("cust", "e").agg(
        F.sum("new_amt").alias("new_amt"),
        _s_if(F.col("scope") == "int", "new_amt").alias("new_int"),
        _s_if(F.col("pw_new") == 1, "new_amt").alias("new_pw"),
        _s_if(F.col("is_hub") == 1, "new_amt").alias("new_hub"),
        F.count(F.lit(1)).alias("n_new"))
    _cn1 = (new_streams.withColumn("e", F.col("e") - 1).groupBy("cust", "e")
            .agg(F.sum("new_amt").alias("new_next")))
    _num = ["lost", "grow", "new_amt", "new_next", "lost_int", "lost_pw", "lost_hub", "grow_int", "grow_pw",
            "grow_hub", "new_int", "new_pw", "new_hub", "n_broken", "n_grew", "n_new"] + [f"lost_{t}" for t in LOSS_TYPES]
    _cm = (_cr.join(_cn, ["cust", "e"], "full").join(_cn1, ["cust", "e"], "full").na.fill(0, _num)
           .join(elig.withColumnRenamed("mi", "e"), ["cust", "e"], "inner")
           .withColumn("gain_tot", F.col("grow") + F.col("new_amt"))
           .withColumn("net_loss", F.col("lost") - F.col("grow") - F.col("new_amt") - F.col("new_next"))
           .withColumn("net_gain", F.col("gain_tot") - F.col("lost"))
           .withColumn("direction", F.when(F.col("net_loss") >= TIER0 * F.col("base_in"), F.lit("loss"))
                       .when(F.col("net_gain") >= TIER0 * F.col("base_in"), F.lit("gain")))
           .where(F.col("direction").isNotNull()))
    _isl = F.col("direction") == "loss"
    _top = F.greatest(*[F.col(f"lost_{t}") for t in LOSS_TYPES])
    _ltype = F.lit("open")
    for t in reversed(LOSS_TYPES):
        _ltype = F.when(F.col(f"lost_{t}") == _top, F.lit(t)).otherwise(_ltype)
    _sh = (_cm
           .withColumn("amount", F.when(_isl, F.col("net_loss")).otherwise(F.col("net_gain")))
           .withColumn("size", F.col("amount") / F.col("base_in"))
           .withColumn("type", F.when(_isl, _ltype).when(F.col("new_amt") >= 0.5 * F.col("gain_tot"), F.lit("new payer"))
                       .otherwise(F.lit("existing payer grew")))
           .withColumn("source", F.when(F.when(_isl, F.col("lost_int") >= 0.5 * F.col("lost"))
                                        .otherwise(F.col("grow_int") + F.col("new_int") >= 0.5 * F.col("gain_tot")),
                                        F.lit("PNC customer payer")).otherwise(F.lit("external payer")))
           .withColumn("breadth", F.when(F.when(_isl, F.col("lost_pw") >= 0.5 * F.col("lost"))
                                         .otherwise(F.col("grow_pw") + F.col("new_pw") >= 0.5 * F.col("gain_tot")),
                                         F.lit("payer-wide")).otherwise(F.lit("single-link")))
           .withColumn("hub", F.when(F.when(_isl, F.col("lost_hub") >= 0.5 * F.col("lost"))
                                     .otherwise(F.col("grow_hub") + F.col("new_hub") >= 0.5 * F.col("gain_tot")),
                                     F.lit("hub payer")).otherwise(F.lit("non-hub payer"))))
    _wc = W.partitionBy("cust").orderBy("e")
    _sh = (_sh.withColumn("prev_e", F.lag("e").over(_wc))
              .withColumn("clean", F.col("prev_e").isNull() | (F.col("e") - F.col("prev_e") > CLEAN)))
    shocks = cache_parquet(_sh, "cust_events")

    _ev = (shocks.where(F.col("clean") & F.col("e").between(EV_START, EV_END))
           .withColumn("rn", F.row_number().over(W.partitionBy("cust", "direction").orderBy("e")))
           .where(F.col("rn") == 1)
           .select("cust", "customer_pwr_id", "master", "customer_name", "naics2n", "e", "base_in", "pbin",
                   "segment", "size_band", "direction", "size", "amount", "type", "source", "breadth", "hub",
                   "lost", "gain_tot"))
    events = _ev.toPandas()

    _st = risk.agg(F.count(F.lit(1)).alias("standing"),
                   F.sum(F.col("gross_break").cast("long")).alias("gross_breaks"),
                   F.sum(F.col("broke").cast("long")).alias("net_breaks"),
                   F.sum(F.col("grew").cast("long")).alias("grows")).toPandas().iloc[0]
    _bt = (stream_events.where(F.col("broke")).groupBy("ev").agg(F.count(F.lit(1)).alias("streams"),
                                                                  F.sum("lost").alias("dollars")).toPandas()
           .rename(columns={"ev": "type"}))
    _gt = stream_events.where(F.col("grew")).agg(F.count(F.lit(1)).alias("streams"), F.sum("gain").alias("dollars")).toPandas()
    _nt = new_streams.agg(F.count(F.lit(1)).alias("streams"), F.sum("new_amt").alias("dollars")).toPandas()
    _ndir = shocks.groupBy("direction").count().toPandas().set_index("direction")["count"]

if len(events) == 0:
    raise ValueError("No events found. Check the pair table values of dir / scope and the thresholds.")
_edges = CONFIG["TIER_EDGES"]
_tlabels = [f"{int(a * 100)}-{int(b * 100)}%" for a, b in zip(_edges[:-1], _edges[1:])] + [f"{int(_edges[-1] * 100)}%+"]
events["tier"] = pd.cut(events["size"], bins=_edges + [np.inf], labels=_tlabels, right=False).astype(str)
events["month"] = events["e"].map(mi2m)
events["evid"] = events["cust"].astype(str) + "|" + events["direction"]

_bt = _bt.set_index("type").reindex(LOSS_TYPES).fillna(0).reset_index()
_types = pd.concat([_bt.assign(kind="break"),
                    pd.DataFrame([dict(type="grow", streams=_gt.streams.iloc[0], dollars=_gt.dollars.iloc[0], kind="gain"),
                                  dict(type="new payer", streams=_nt.streams.iloc[0], dollars=_nt.dollars.iloc[0], kind="gain")])])
_types["streams"] = _types["streams"].fillna(0).astype(int)
_gb, _nb = int(_st.gross_breaks), int(_st.net_breaks)
_sum2 = pd.DataFrame([dict(
    standing_stream_months=int(_st.standing), breaks_before_netting=_gb, breaks_after_netting=_nb,
    share_removed_as_timing_shift=1 - _nb / max(_gb, 1), grows=int(_st.grows),
    loss_event_months=int(_ndir.get("loss", 0)), gain_event_months=int(_ndir.get("gain", 0)),
    first_loss_events=int((events.direction == "loss").sum()), first_gain_events=int((events.direction == "gain").sum()),
    event_months=f"{mi2m(EV_START)} .. {mi2m(EV_END)}")])
CELL.show(("Streams and events", _sum2), ("Stream events by type", _types[["kind", "type", "streams", "dollars"]]))
ans("Stream breaks removed as timing shifts", f"{1 - _nb / max(_gb, 1):.1%} of {_gb:,}")

## §3 · Pre-flight counts, gates and headline

**Headline**, per direction: operating businesses, non-hub payers, payer-wide, size 20%+. If fewer than 1,000
such events (or payer-wide under 10% of operating non-hub events at 20%+), the payer-wide condition is
dropped; if still under 1,000, the size floor drops to 10%. Figures f01 and f02 are saved here.

In [ ]:
CELL.start("§3 · Pre-flight counts, gates, headline")
HEAD, HEAD_LABEL = {}, {}
events["headline"] = False
for d in DIRS:
    E = events[events.direction == d]
    base = (E.segment == "operating") & (E.hub == "non-hub payer")
    big = base & (E["size"] >= CONFIG["HEADLINE_MIN"])
    pw = big & (E.breadth == "payer-wide")
    pw_share = float(pw.sum() / max(big.sum(), 1))
    if pw.sum() >= CONFIG["GATE_MIN_EVENTS"] and pw_share >= CONFIG["GATE_MIN_PAYERWIDE"]:
        sel, lab = pw, f"operating, non-hub, payer-wide, {CONFIG['HEADLINE_MIN']:.0%}+"
    elif big.sum() >= CONFIG["GATE_MIN_EVENTS"]:
        sel, lab = big, f"operating, non-hub, {CONFIG['HEADLINE_MIN']:.0%}+"
    else:
        sel, lab = base, f"operating, non-hub, {TIER0:.0%}+"
    gate(f"{d}: payer-wide headline has enough events", bool(sel is pw),
         f"{int(pw.sum()):,} payer-wide ({pw_share:.1%} of {int(big.sum()):,} operating non-hub at {CONFIG['HEADLINE_MIN']:.0%}+)",
         f"headline = {lab}")
    events.loc[E.index[sel.to_numpy()], "headline"] = True
    HEAD_LABEL[d] = lab
    ans(f"Headline group, {d}", f"{lab}: {int(sel.sum()):,} events")
    _mc = E.groupby("month").size()
    gate(f"{d}: events spread over months", bool(len(_mc) and _mc.max() <= 3 * _mc.median()),
         f"largest month {int(_mc.max()) if len(_mc) else 0:,}, median {float(_mc.median()) if len(_mc) else 0:,.0f}",
         "a month far above the rest points to a data or definition artefact")

_by_month = (events.pivot_table(index="month", columns=["direction", "type"], values="cust", aggfunc="count", fill_value=0))
_by_month.columns = [f"{a}: {b}" for a, b in _by_month.columns]
_by_month = _by_month.reset_index()
_tier_tab = (events.pivot_table(index=["direction", "tier"], columns="breadth", values="cust", aggfunc="count", fill_value=0)
             .reset_index())
_cuts = pd.concat([events.groupby(["direction", c]).agg(events=("cust", "count"), headline=("headline", "sum"),
                                                        median_size=("size", "median"), dollars=("amount", "sum"))
                   .reset_index().rename(columns={c: "value"}).assign(cut=c)
                   for c in ["type", "source", "breadth", "hub", "segment", "size_band"]])
_cuts = _cuts[["direction", "cut", "value", "events", "headline", "median_size", "dollars"]]

# f01: events per month by type, one panel per direction
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4), sharey=False)
for ax, d in zip(axes, DIRS):
    E = events[events.direction == d]
    tps = [t for t in (LOSS_TYPES if d == "loss" else GAIN_TYPES) if t in set(E["type"])]
    pv = E.pivot_table(index="month", columns="type", values="cust", aggfunc="count", fill_value=0).reindex(columns=tps).fillna(0)
    bottom = np.zeros(len(pv))
    for t in tps:
        ax.bar(pv.index, pv[t], bottom=bottom, color=TYPE_COLORS[t], label=t, width=0.78, edgecolor=SURF, linewidth=1.2)
        bottom += pv[t].to_numpy()
    ax.set_title(f"{d.capitalize()} events per month (first per customer)", pad=26)
    ax.set_ylabel("customers"); ax.grid(axis="x", visible=False); ax.tick_params(axis="x", rotation=60)
    ax.legend(ncol=len(tps), loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=9)
fig.tight_layout()
savefig(fig, "f01_events_by_month", "3", "Events per month by type, losses and gains")

# f02: event size by direction
fig, ax = plt.subplots(figsize=(8.5, 4.2))
_bins = np.linspace(TIER0, 1.5, 29)
for d in DIRS:
    ax.hist(events.loc[events.direction == d, "size"].clip(upper=1.5), bins=_bins, histtype="step",
            color=C_DIR[d], linewidth=2, label=d)
for _x in _edges[1:]:
    ax.axvline(_x, color=INK2, linewidth=1, linestyle=":")
ax.set_title("Event size: net dollars / baseline monthly inflow (capped at 150%)")
ax.set_xlabel("event size"); ax.set_ylabel("customers"); ax.legend(loc="lower right", bbox_to_anchor=(1, 1), ncol=2)
ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0, decimals=0))
savefig(fig, "f02_event_size", "3", "Distribution of event size by direction; dotted lines are the tier edges")

CELL.show(("Gates", gates_frame()), ("Events by tier and breadth", _tier_tab), ("Events by cut", _cuts),
          ("Events by month and type", _by_month))
RUN_FULL = CONFIG["RUN_STAGE"] == "full"
if not RUN_FULL:
    CELL.log("RUN_STAGE = preflight: stopping here. Set RUN_STAGE = 'full' to continue (work files are reused).")

## §4 · Matched controls and the event panel

Events are sampled per stratum (direction x headline / other payer-wide / single-link), up to 5,000 each,
by a stable hash. Up to 3 controls per event: same month, NAICS2 and baseline-inflow decile, nearest on
baseline inflow, a different master, no event of either direction in the 6 months before or in the event
month. Controls are free to have events later, like any customer. Months past the end of data stay empty;
months inside the data with no metrics row count as zero activity.

In [ ]:
if RUN_FULL:
    CELL.start("§4 · Matched controls and the event panel")
    PRE, POST, NC = CONFIG["PRE_MONTHS"], CONFIG["POST_MONTHS"], CONFIG["N_CONTROLS"]
    events["stratum"] = events["direction"] + ": " + np.where(
        events["headline"], "headline", np.where(events["breadth"] == "payer-wide", "other payer-wide", "single-link"))
    events["_h"] = events["evid"].map(lambda s: int(hashlib.md5(str(s).encode()).hexdigest()[:8], 16))
    _pe = (events.sort_values("_h").groupby("stratum", group_keys=False).head(CONFIG["PANEL_PER_STRATUM"])
           .reset_index(drop=True))
    with step("matching controls"):
        _evs = spark.createDataFrame(
            _pe[["evid", "cust", "master", "e", "naics2n", "pbin", "base_in"]]
            .rename(columns={"cust": "ev_cust", "master": "ev_master", "base_in": "ev_base"})
            .astype({"e": "int32", "pbin": "int32", "ev_base": "float64"}))
        _arr = shocks.groupBy("cust").agg(F.collect_set("e").alias("shock_arr"))
        _cand = (elig.select(F.col("cust").alias("unit"), F.col("master").alias("u_master"),
                             F.col("mi").alias("e"), "naics2n", "pbin", F.col("base_in").alias("ubase"))
                 .join(_arr.withColumnRenamed("cust", "unit"), "unit", "left")
                 .withColumn("shock_arr", F.coalesce(F.col("shock_arr"), F.array().cast("array<int>"))))
        _dec = lambda c: F.ceil(F.col(c) / 10.0).cast("int")
        _m = (_evs.withColumn("dec", _dec("pbin")).drop("pbin")
              .join(_cand.withColumn("dec", _dec("pbin")).drop("pbin"), ["e", "naics2n", "dec"], "inner")
              .where((F.col("unit") != F.col("ev_cust")) & (F.col("u_master") != F.col("ev_master"))
                     & F.expr(f"NOT exists(shock_arr, x -> x - e >= -{CLEAN} AND x - e <= {CONFIG['CONTROL_CLEAN_AFTER']})"))
              .withColumn("dist", F.abs(F.log(F.col("ubase")) - F.log(F.col("ev_base"))))
              .withColumn("rn", F.row_number().over(W.partitionBy("evid").orderBy("dist", F.xxhash64("unit", "evid"))))
              .where(F.col("rn") <= NC)
              .select("evid", "e", "unit", "ubase", F.lit("C").alias("role")))
        _units = (_evs.select("evid", "e", F.col("ev_cust").alias("unit"), F.col("ev_base").alias("ubase"),
                              F.lit("S").alias("role")).unionByName(_m))
        _ukey = hashlib.md5(("|".join(sorted(_pe["evid"])) + f"{NC}{CONFIG['CONTROL_CLEAN_AFTER']}").encode()).hexdigest()[:8]
        units = cache_parquet(_units, f"units_{_ukey}")

    with step("event panel (months -6 .. +6): payment, deposit, revenue and credit outcomes"):
        _grid = (units.withColumn("t", F.explode(F.sequence(F.lit(-PRE), F.lit(POST))))
                 .withColumn("mi", F.col("e") + F.col("t"))
                 .where((F.col("mi") >= 0) & (F.col("mi") <= MAX_MI)))
        _pan = _grid.join(pop.select(F.col("cust").alias("unit"), "mi", *[F.col(c).cast("float").alias(c) for c in OUTCOMES]),
                          ["unit", "mi"], "left")
        EXTRA = {}          # output column -> dict(label, kind, mult, family, subset, mask)
        if EXTRA_ACTIVE:
            _ids = (pop.where(F.col("customer_pwr_id").isNotNull())
                    .select(F.col("cust").alias("unit"), "customer_pwr_id").dropDuplicates(["unit"])
                    .join(units.select("unit").distinct(), "unit", "left_semi"))
            for _x in EXTRA_ACTIVE:
                _key = "unit" if _x["id_kind"] == "mdm_id" else "customer_pwr_id"
                _mc = F.col(_x["month_col"])
                _mstr = (F.when(_mc.cast("string").rlike(r"^\d{4}-\d{2}$"), _mc.cast("string"))
                         .otherwise(F.date_format(_mc.cast("date"), "yyyy-MM")))
                _cols = {out: m["src"] for out, m in _x["measures"].items()}
                if _x.get("mask"):
                    _cols[f"_mask_{_x['name']}"] = _x["mask"]["col"]
                _xt = (spark.table(_x["table"])
                       .select(F.col(_x["id_col"]).cast("string").alias(_key), mi_col(_mstr).alias("mi"),
                               *[F.col(src).cast("double").alias(out) for out, src in _cols.items()])
                       .where(F.col("mi").between(0, MAX_MI))
                       .groupBy(_key, "mi").agg(*[F.avg(out).cast("float").alias(out) for out in _cols]))
                if _key == "customer_pwr_id":
                    _xt = _xt.join(_ids, "customer_pwr_id", "inner").drop("customer_pwr_id")
                _pan = _pan.join(_xt, ["unit", "mi"], "left")
                for out, m in _x["measures"].items():
                    EXTRA[out] = dict(label=m["label"], kind=m["kind"], mult=m.get("mult", 1.0), family=_x["name"],
                                      subset=_x.get("subset_pos"),
                                      mask=(f"_mask_{_x['name']}", [m2mi(v) for v in _x["mask"]["months"]]) if _x.get("mask") else None)
        panel = _pan.toPandas()

    _nctl = panel[panel.role == "C"].groupby("evid")["unit"].nunique().reindex(_pe["evid"]).fillna(0).astype(int)
    _pe["n_controls"] = _nctl.to_numpy()
    _match = _pe.groupby(["stratum", "n_controls"]).size().rename("events").reset_index()
    _n_all = len(_pe)
    _pe = _pe[_pe["n_controls"] > 0].reset_index(drop=True)
    _ub = panel.drop_duplicates(["evid", "role", "unit"])
    _bal = (_ub.merge(_pe[["evid", "stratum"]], on="evid").groupby(["stratum", "role"])
            .agg(units=("unit", "nunique"), median_base_in=("ubase", "median")).reset_index()
            .pivot(index="stratum", columns="role", values=["units", "median_base_in"]))
    _bal.columns = [f"{a}_{'event' if b == 'S' else 'controls'}" for a, b in _bal.columns]
    _bal = _bal.reset_index()
    gate("events with at least one control", len(_pe) >= 0.8 * max(_n_all, 1), f"{len(_pe):,} of {_n_all:,}",
         "unmatched events are left out of the effects")
    CELL.show(("Panel by stratum (events, controls, median baseline inflow)", _bal),
              ("Panel", pd.DataFrame([dict(events=len(_pe), panel_rows=len(panel), outcome_columns=len(OUTCOMES) + len(EXTRA))])),
              ("Controls found per event", _match))

## §5 · Effects

Every unit is measured against its own baseline (months -6 to -1). The effect is the event group's average
minus its matched controls' average, month by month.

- **logpct** (dollars, counts, balances, revenue, DHE): log change from baseline, capped at x10 either way;
  group averages are converted back to % change, and the effect is % relative to controls.
- **diff** (shares, flags, days, PD): change from baseline, in the column's own units (shares in pp).
- **scaled** (net flow, balance changes): change from baseline / baseline monthly inflow, in pp.
- Credit outcomes only for customers with DHE above zero at baseline; deposit outcomes leave out customers
  whose new accounts appear in the acquisition months.
- Intervals: cluster bootstrap by master (an event family counts once).
- **Pass-through** = log change in outflow / log change in inflow at month +3: the % of an inflow change the
  customer passes on to its own payees.
- **Pre-trend check:** baseline moved to months -6..-4, effect read at -3..-1. It should sit on zero.

In [ ]:
if RUN_FULL:
    CELL.start("§5 · Effects")
    ALL_OUT = {c: dict(label=v[0], kind=v[1], mult=v[2], family="payments", subset=None, mask=None)
               for c, v in OUTCOMES.items()}
    ALL_OUT.update(EXTRA)
    TS = np.arange(-PRE, POST + 1)
    ET, B, LCAP = CONFIG["EFFECT_T"], CONFIG["BOOT_REPS"], math.log(CONFIG["LOG_CAP"])
    _meta = _pe.set_index("evid")
    _keys = panel[["evid", "role", "unit"]].drop_duplicates()
    _midx = pd.MultiIndex.from_frame(_keys)
    _ubase = panel.drop_duplicates(["evid", "role", "unit"]).set_index(["evid", "role", "unit"])["ubase"]
    _wide_cache = {}
    _pidx = pd.MultiIndex.from_frame(panel[["evid", "role", "unit", "t"]])
    _SUBSET_COLS = {o["subset"] for o in ALL_OUT.values() if o["subset"]}

    def _wide(col):
        """units x months matrix of the raw value (core payment outcomes: missing month = 0)."""
        if col in _wide_cache:
            return _wide_cache[col]
        v = panel[col].astype("float64")
        if col in OUTCOMES:
            v = v.fillna(0.0)
        m = pd.Series(v.to_numpy(), index=_pidx).unstack("t").reindex(index=_midx, columns=TS)
        if col in _SUBSET_COLS:
            _wide_cache[col] = m
        return m

    def _matrix(col, base_ts):
        o = ALL_OUT[col]
        m = _wide(col) * o["mult"]
        base = m[base_ts].mean(axis=1, skipna=True)
        if o["kind"] == "logpct":
            out = np.log((m.clip(lower=0) + 1.0).div(base.clip(lower=0) + 1.0, axis=0)).clip(-LCAP, LCAP)
        elif o["kind"] == "scaled":
            out = m.sub(base, axis=0).div(_ubase.reindex(m.index), axis=0).mul(100.0).clip(-500, 500)
        else:
            out = m.sub(base, axis=0)
        if o["subset"]:
            sb = _wide(o["subset"])[base_ts].mean(axis=1, skipna=True)
            out.loc[~(sb > 0).to_numpy()] = np.nan
        if o["mask"]:
            mc, mis = o["mask"]
            mm = panel[panel["mi"].isin(mis) & (panel[mc] > 0)][["evid", "role", "unit"]].drop_duplicates()
            if len(mm):
                out.loc[out.index.isin(pd.MultiIndex.from_frame(mm))] = np.nan
        return out

    def _sc(col, base_ts):
        m = _matrix(col, base_ts)
        S = m.xs("S", level="role").droplevel("unit")
        C = m.xs("C", level="role").droplevel("unit")
        return S, C

    def _effect(S, C, ids, gkey):
        ii = S.index.intersection(pd.Index(ids))
        if len(ii) < 20:
            return None
        Cg = C[C.index.isin(ii)]
        s, c = S.loc[ii].to_numpy(float), Cg.to_numpy(float)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ms, mc = np.nanmean(s, axis=0), np.nanmean(c, axis=0)
        uniq, cs = np.unique(_meta.loc[ii, "master"].astype(str).to_numpy(), return_inverse=True)
        cc = np.searchsorted(uniq, _meta.loc[Cg.index, "master"].astype(str).to_numpy())
        K, T = len(uniq), s.shape[1]
        Ss, Ns, Sc, Nc = (np.zeros((K, T)) for _ in range(4))
        np.add.at(Ss, cs, np.nan_to_num(s)); np.add.at(Ns, cs, ~np.isnan(s))
        np.add.at(Sc, cc, np.nan_to_num(c)); np.add.at(Nc, cc, ~np.isnan(c))
        rng = np.random.default_rng(CONFIG["SEED"] + int(hashlib.md5(str(gkey).encode()).hexdigest()[:6], 16))
        Wt = rng.multinomial(K, np.full(K, 1.0 / K), size=B).astype(float)
        with np.errstate(all="ignore"):
            bd = (Wt @ Ss) / (Wt @ Ns) - (Wt @ Sc) / (Wt @ Nc)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            lo, hi = np.nanpercentile(bd, 2.5, axis=0), np.nanpercentile(bd, 97.5, axis=0)
        return dict(ms=ms, mc=mc, d=ms - mc, lo=lo, hi=hi, n=np.sum(~np.isnan(s), axis=0), bd=bd)

    def _disp(kind, v):
        return 100.0 * (np.exp(v) - 1.0) if kind == "logpct" else v

    UNIT = {"logpct": "%", "diff": "change", "scaled": "pp of inflow"}

    def _ttl(label, kind):
        return label if "(" in label else f"{label} ({UNIT[kind]})"

    GROUPS, GCUT = {}, {}
    for d in DIRS:
        E = _pe[_pe.direction == d]
        GROUPS[(d, "headline")] = E.loc[E.headline, "evid"]; GCUT[(d, "headline")] = "headline"
        GROUPS[(d, "all in panel")] = E["evid"]; GCUT[(d, "all in panel")] = "headline"
        for _c, _pref, _cut in [("tier", "tier ", "tier"), ("type", "", "type"), ("source", "", "cut"),
                                ("breadth", "", "cut"), ("hub", "", "cut"), ("segment", "", "segment"),
                                ("size_band", "size ", "size")]:
            for _v in sorted(E[_c].unique()):
                GROUPS[(d, f"{_pref}{_v}")] = E.loc[E[_c] == _v, "evid"]; GCUT[(d, f"{_pref}{_v}")] = _cut

    _rows, _pre_rows, BOOT = [], [], {}
    with step(f"effects: {len(ALL_OUT)} outcomes x {len(GROUPS)} groups, {B} bootstrap draws"):
        for col, o in ALL_OUT.items():
            S, C = _sc(col, list(range(-PRE, 0)))
            for g, ids in GROUPS.items():
                r = _effect(S, C, ids, g)
                if r is None:
                    continue
                if col in ("amt_all_in", "amt_all_out"):
                    BOOT[(g, col)] = (r["d"], r["bd"])
                for j, t in enumerate(TS):
                    _rows.append(dict(direction=g[0], group=g[1], cut=GCUT[g], outcome=col, label=o["label"],
                                      family=o["family"], kind=o["kind"], t=int(t), n=int(r["n"][j]),
                                      event=_disp(o["kind"], r["ms"][j]), control=_disp(o["kind"], r["mc"][j]),
                                      effect=_disp(o["kind"], r["d"][j]), lo=_disp(o["kind"], r["lo"][j]),
                                      hi=_disp(o["kind"], r["hi"][j])))
            Sp, Cp = _sc(col, list(range(-PRE, -3)))
            for d in DIRS:
                r = _effect(Sp, Cp, GROUPS[(d, "headline")], (d, "pre", col))
                if r is None:
                    continue
                for t in (-3, -2, -1):
                    j = int(np.where(TS == t)[0][0])
                    _pre_rows.append(dict(direction=d, outcome=col, label=o["label"], kind=o["kind"], t=t, n=int(r["n"][j]),
                                          effect=_disp(o["kind"], r["d"][j]), lo=_disp(o["kind"], r["lo"][j]),
                                          hi=_disp(o["kind"], r["hi"][j])))
    effects = pd.DataFrame(_rows)
    pretrend = pd.DataFrame(_pre_rows)

    # pass-through: log outflow effect / log inflow effect at month +3, same bootstrap draws
    _j = int(np.where(TS == ET)[0][0])
    _pt = []
    for g in GROUPS:
        if (g, "amt_all_in") not in BOOT or (g, "amt_all_out") not in BOOT:
            continue
        din, bin_ = BOOT[(g, "amt_all_in")]; dout, bout = BOOT[(g, "amt_all_out")]
        with np.errstate(all="ignore"):
            rb = bout[:, _j] / bin_[:, _j]
        ok_in = (np.nanpercentile(bin_[:, _j], 2.5) > 0) or (np.nanpercentile(bin_[:, _j], 97.5) < 0)
        _pt.append(dict(direction=g[0], group=g[1], cut=GCUT[g], events=len(GROUPS[g]),
                        inflow_pct=_disp("logpct", din[_j]), outflow_pct=_disp("logpct", dout[_j]),
                        pass_through=(dout[_j] / din[_j]) if ok_in else np.nan,
                        lo=np.nanpercentile(rb, 2.5) if ok_in else np.nan,
                        hi=np.nanpercentile(rb, 97.5) if ok_in else np.nan))
    passthru = pd.DataFrame(_pt)

    effects.to_csv(os.path.join(OUT_DIR, "p1_effects.csv"), index=False)
    pretrend.to_csv(os.path.join(OUT_DIR, "p1_pretrend.csv"), index=False)
    passthru.to_csv(os.path.join(OUT_DIR, "p1_pass_through.csv"), index=False)
    _pe.drop(columns=["_h"]).to_parquet(os.path.join(OUT_DIR, "p1_events.parquet"), index=False)

    KEY = [c for c in ["amt_all_in", "amt_all_out", "ntxn_all_out", "ncpty_ext_out", "new_cpty_ext_out",
                       "rec_broken_share_out", "net_flow_all", "dep_bal_mean", "dep_bal_vs_inflow", "days_negative",
                       "tm_rev_12m", "gross_rev_12m", "pd_pct_now", "past_due_days"] if c in ALL_OUT]

    def _event_fig(cols, d, name, title):
        """Left: event customers and controls. Right: the difference with its interval. One row per outcome."""
        e = effects[(effects.direction == d) & (effects.group == "headline") & effects.outcome.isin(cols)]
        cols = [c for c in cols if c in set(e.outcome)]
        if not cols:
            return
        fig, axes = plt.subplots(len(cols), 2, figsize=(11, 3.3 * len(cols)), squeeze=False)
        for r, col in enumerate(cols):
            x = e[e.outcome == col].sort_values("t")
            a, b = axes[r]
            a.plot(x.t, x.control, color=C_CTRL, marker="o", markersize=4, label="matched controls")
            a.plot(x.t, x.event, color=C_DIR[d], marker="o", markersize=4, label=f"{d} event")
            a.axvline(0, color=INK2, linewidth=1, linestyle=":")
            a.set_title(x.label.iloc[0]); a.set_ylabel(f"change from baseline ({UNIT[x.kind.iloc[0]]})")
            a.legend(loc="best")
            b.fill_between(x.t, x.lo, x.hi, color=C_DIR[d], alpha=0.18, linewidth=0)
            b.plot(x.t, x.effect, color=C_DIR[d], marker="o", markersize=4)
            b.axhline(0, color=INK2, linewidth=1); b.axvline(0, color=INK2, linewidth=1, linestyle=":")
            b.set_title("Event minus controls (95% interval)"); b.set_ylabel(UNIT[x.kind.iloc[0]])
            for ax_ in (a, b):
                ax_.set_xticks(TS)
            if r == len(cols) - 1:
                a.set_xlabel("months from the event"); b.set_xlabel("months from the event")
        fig.suptitle(f"{title} — {d}: {HEAD_LABEL[d]}, n = {int(e.n.max()):,}", x=0.01, ha="left",
                     fontsize=12, fontweight="bold")
        fig.tight_layout(rect=(0, 0, 1, 0.97))
        savefig(fig, name, "5", f"{title}, {d} events against matched controls, months -{PRE} to +{POST}")
        return e, cols

    def _event_html(cols, d, name, title):
        try:
            import plotly.graph_objects as go
            from plotly.subplots import make_subplots
        except ImportError:
            return
        e = effects[(effects.direction == d) & (effects.group == "headline") & effects.outcome.isin(cols)]
        cols = [c for c in cols if c in set(e.outcome)]
        if not cols:
            return
        rgb = {"loss": "42,120,214", "gain": "235,104,52"}[d]
        pf = make_subplots(rows=len(cols), cols=2, subplot_titles=sum(
            [[e[e.outcome == c].label.iloc[0], "Event minus controls"] for c in cols], []))
        for r, col in enumerate(cols, start=1):
            x = e[e.outcome == col].sort_values("t")
            pf.add_trace(go.Scatter(x=x.t, y=x.control, name="matched controls", mode="lines+markers",
                                    line=dict(color=C_CTRL), showlegend=(r == 1)), row=r, col=1)
            pf.add_trace(go.Scatter(x=x.t, y=x.event, name=f"{d} event", mode="lines+markers",
                                    line=dict(color=C_DIR[d]), showlegend=(r == 1)), row=r, col=1)
            pf.add_trace(go.Scatter(x=list(x.t) + list(x.t[::-1]), y=list(x.hi) + list(x.lo[::-1]), fill="toself",
                                    fillcolor=f"rgba({rgb},0.18)", line=dict(width=0), hoverinfo="skip",
                                    showlegend=False), row=r, col=2)
            pf.add_trace(go.Scatter(x=x.t, y=x.effect, mode="lines+markers", line=dict(color=C_DIR[d]), showlegend=False,
                                    customdata=np.stack([x.lo, x.hi, x.n], axis=1),
                                    hovertemplate="month %{x}: %{y:.2f} (%{customdata[0]:.2f} to %{customdata[1]:.2f}), n=%{customdata[2]:,}<extra></extra>"),
                         row=r, col=2)
        pf.update_layout(title=f"{title} — {d}: {HEAD_LABEL[d]}", template="simple_white", hovermode="x unified",
                         height=330 * len(cols), paper_bgcolor=SURF, plot_bgcolor=SURF)
        save_plotly(pf, name, "5", f"{title}, {d}")

    def _dot_fig(d, cuts, name, title, caption, groups=None):
        sub = effects[(effects.direction == d) & (effects.t == ET) & effects.cut.isin(cuts)]
        groups = groups or [g for (dd, g) in GROUPS if dd == d and GCUT[(dd, g)] in cuts and g in set(sub.group)]
        cols = [c for c in KEY if c in set(sub.outcome)]
        if not cols or not groups:
            return
        ncol = 3; nrow = int(math.ceil(len(cols) / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(13.5, 0.9 + (0.36 * len(groups) + 1.4) * nrow), squeeze=False)
        for k, col in enumerate(cols):
            ax = axes[k // ncol][k % ncol]
            x = sub[sub.outcome == col].set_index("group").reindex(groups)
            y = np.arange(len(groups))[::-1]
            ax.hlines(y, x.lo, x.hi, color=C_DIR[d], linewidth=2, alpha=0.55)
            ax.plot(x.effect, y, "o", color=C_DIR[d], markersize=7, markeredgecolor=SURF, markeredgewidth=1.5)
            ax.axvline(0, color=INK2, linewidth=1)
            ax.set_yticks(y); ax.set_yticklabels(groups if k % ncol == 0 else [])
            lab = x.label.dropna().iloc[0] if x.label.notna().any() else col
            knd = x.kind.dropna().iloc[0] if x.kind.notna().any() else "diff"
            ax.set_title(_ttl(lab, knd), fontsize=9.5)
            ax.grid(axis="y", visible=False)
        for k in range(len(cols), nrow * ncol):
            axes[k // ncol][k % ncol].axis("off")
        fig.suptitle(title, x=0.01, ha="left", fontsize=12, fontweight="bold")
        fig.tight_layout(rect=(0, 0, 1, 0.96))
        savefig(fig, name, "5", caption)

    with step("figures"):
        for d in DIRS:
            _event_fig(["amt_all_in"], d, f"f03_inflow_{d}", "Incoming dollars around the event")
            _event_html(["amt_all_in"], d, f"f03_inflow_{d}", "Incoming dollars around the event")
            _event_fig(["amt_all_out", "ntxn_all_out", "net_flow_all"], d, f"f04_outflow_{d}", "Own outgoing payments around the event")
            _event_html(["amt_all_out", "ntxn_all_out", "net_flow_all"], d, f"f04_outflow_{d}", "Own outgoing payments around the event")
            _event_fig(["rec_broken_share_out", "rec_broken_amt_share_out", "new_cpty_ext_out", "lost_cpty_ext_out"], d,
                       f"f05_own_payees_{d}", "Own payees around the event")
            for fam, fn, tt in [("deposits", "f06_deposits", "Deposits around the event"),
                                ("credit", "f08_credit", "Credit around the event (customers with DHE)"),
                                ("revenue", "f09_revenue", "Revenue around the event")]:
                _fc = [c for c, o in ALL_OUT.items() if o["family"] == fam]
                if _fc:
                    _event_fig(_fc, d, f"{fn}_{d}", tt)
                    if fam == "deposits":
                        _event_html(_fc, d, f"{fn}_{d}", tt)
            _dot_fig(d, ["tier"], f"f10_dose_response_{d}",
                     f"{d.capitalize()}: effect at month +{ET} by event size (event minus controls, 95% interval)",
                     f"Dose response, {d}: effect at month +{ET} by event size",
                     groups=[f"tier {l}" for l in _tlabels if (d, f"tier {l}") in GROUPS])
            _dot_fig(d, ["type", "cut", "segment", "size"], f"f11_cuts_{d}",
                     f"{d.capitalize()}: effect at month +{ET} by type, source, breadth, hub, segment and size",
                     f"Cuts, {d}: effect at month +{ET}")
            if not pretrend.empty and (pretrend.direction == d).any():
                _p = pretrend[pretrend.direction == d]
                fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
                for ax, col in zip(axes, [c for c in ["amt_all_in", "amt_all_out", "rec_broken_share_out"] if c in set(_p.outcome)]):
                    x = _p[_p.outcome == col].sort_values("t")
                    ax.fill_between(x.t, x.lo, x.hi, color=C_DIR[d], alpha=0.18, linewidth=0)
                    ax.plot(x.t, x.effect, color=C_DIR[d], marker="o")
                    ax.axhline(0, color=INK2, linewidth=1); ax.set_xticks([-3, -2, -1])
                    ax.set_title(_ttl(x.label.iloc[0], x.kind.iloc[0]), fontsize=9.5)
                fig.suptitle(f"{d.capitalize()} pre-trend check: baseline months -6..-4, effect before the event (should sit on zero)",
                             x=0.01, ha="left", fontsize=11, fontweight="bold")
                fig.tight_layout(rect=(0, 0, 1, 0.9))
                savefig(fig, f"f15_pretrend_{d}", "5", f"Pre-trend check, {d} headline")

        # f17: pass-through by group, both directions
        _ptg = ["headline", "all in panel", "size small", "size mid", "size large", "operating", "finance & public",
                "PNC customer payer", "external payer", "payer-wide", "single-link"]
        _pp = passthru[passthru.group.isin(_ptg)]
        if not _pp.empty:
            fig, ax = plt.subplots(figsize=(9, 0.45 * len(_ptg) + 1.6))
            y0 = np.arange(len(_ptg))[::-1]
            for k, d in enumerate(DIRS):
                x = _pp[_pp.direction == d].set_index("group").reindex(_ptg)
                y = y0 + (0.16 if d == "loss" else -0.16)
                ax.hlines(y, 100 * x.lo, 100 * x.hi, color=C_DIR[d], linewidth=2, alpha=0.55)
                ax.plot(100 * x.pass_through, y, "o", color=C_DIR[d], markersize=7, markeredgecolor=SURF,
                        markeredgewidth=1.5, label=d)
            ax.axvline(0, color=INK2, linewidth=1)
            ax.set_yticks(y0); ax.set_yticklabels(_ptg); ax.grid(axis="y", visible=False)
            ax.set_xlabel("% of the inflow change passed on to own outflow (month +3)")
            ax.set_title(f"Pass-through at month +{ET}, with 95% interval")
            ax.legend(loc="lower right", bbox_to_anchor=(1, 1), ncol=2)
            savefig(fig, "f17_pass_through", "5", "Pass-through: % of an inflow change passed on to outflow, by direction and group")

    # tables
    def _fmt_eff(sub):
        return {(r.outcome): ci(r.effect, r.lo, r.hi) for r in sub.itertuples()}

    _lab = {c: _ttl(o["label"], o["kind"]) for c, o in ALL_OUT.items()}
    _t3 = pd.DataFrame({"outcome": [_lab[c] for c in ALL_OUT]})
    for d in DIRS:
        h = effects[(effects.direction == d) & (effects.group == "headline") & (effects.t == ET)].set_index("outcome")
        _t3[f"{d}: n"] = [int(h.n.get(c, 0)) if c in h.index else 0 for c in ALL_OUT]
        _t3[f"{d}: effect (95% interval)"] = [ci(h.effect[c], h.lo[c], h.hi[c]) if c in h.index else "" for c in ALL_OUT]
    _bym = (effects[(effects.group == "headline") & effects.t.isin([-1, 0, 1, 3, 6])]
            .pivot_table(index="outcome", columns=["direction", "t"], values="effect"))
    _bym.columns = [f"{a} t={b:+d}" for a, b in _bym.columns]
    _bym = _bym.reindex([c for c in ALL_OUT if c in _bym.index]).reset_index()
    _bym["outcome"] = _bym["outcome"].map(_lab)
    _dose = (effects[(effects.cut == "tier") & (effects.t == ET)]
             .pivot_table(index="outcome", columns=["direction", "group"], values="effect"))
    _dose.columns = [f"{a}: {b}" for a, b in _dose.columns]
    _dose = _dose.reindex([c for c in KEY if c in _dose.index]).reset_index()
    _dose["outcome"] = _dose["outcome"].map(_lab)
    _ptab = passthru.assign(pass_through=lambda x: x.pass_through.map(lambda v: f"{100 * v:.0f}%" if np.isfinite(v) else ""),
                            interval=lambda x: [f"{100 * a:.0f}% to {100 * b:.0f}%" if np.isfinite(a) else "inflow effect n.s."
                                                for a, b in zip(x.lo, x.hi)])
    _ptab = _ptab[_ptab.group.isin(_ptg)][["direction", "group", "events", "inflow_pct", "outflow_pct", "pass_through", "interval"]]
    _prt = pd.DataFrame({"outcome": [_lab[c] for c in ALL_OUT]})
    for d in DIRS:
        p = pretrend[(pretrend.direction == d) & (pretrend.t == -1)].set_index("outcome") if not pretrend.empty else pd.DataFrame()
        _prt[f"{d}: effect at -1"] = [ci(p.effect[c], p.lo[c], p.hi[c]) if c in p.index else "" for c in ALL_OUT]
        if "amt_all_out" in p.index:
            gate(f"{d}: pre-trend flat for outgoing dollars", bool(p.lo["amt_all_out"] <= 0 <= p.hi["amt_all_out"]),
                 ci(p.effect["amt_all_out"], p.lo["amt_all_out"], p.hi["amt_all_out"]) + " % at month -1",
                 "if it fails, read effects as event plus an earlier drift")
        if "amt_all_in" in p.index:
            gate(f"{d}: pre-trend flat for incoming dollars", bool(p.lo["amt_all_in"] <= 0 <= p.hi["amt_all_in"]),
                 ci(p.effect["amt_all_in"], p.lo["amt_all_in"], p.hi["amt_all_in"]) + " % at month -1",
                 "a spike before the event means timing shifts are still in the events")
    CELL.show((f"Headline effect at month +{ET}, event minus matched controls", _t3),
              ("Headline effect by month", _bym),
              (f"Dose response: effect at month +{ET} by event size", _dose),
              (f"Pass-through at month +{ET}: % of the inflow change passed on to outflow", _ptab),
              ("Pre-trend check at month -1 (should include zero)", _prt))
    for d in DIRS:
        h = effects[(effects.direction == d) & (effects.group == "headline") & (effects.t == ET)].set_index("outcome")
        for c in [c for c in KEY if c in h.index]:
            ans(f"{d} +{ET}: {_lab[c]}", ci(h.effect[c], h.lo[c], h.hi[c]))
        p = passthru[(passthru.direction == d) & (passthru.group == "headline")]
        if len(p) and np.isfinite(p.pass_through.iloc[0]):
            ans(f"{d}: pass-through at +{ET}", f"{100 * p.pass_through.iloc[0]:.0f}% ({100 * p.lo.iloc[0]:.0f}% to {100 * p.hi.iloc[0]:.0f}%)")

## §6 · Second hop

Does it pass on? For event customers and their controls that pay other PNC customers through standing
streams: the share of those streams that break (losses) or grow (gains), month by month. Read from the
receivers' side, so this is the next customer's own event.

In [ ]:
if RUN_FULL:
    CELL.start("§6 · Second hop")
    with step("standing internal streams paid by event customers and controls"):
        _u = units.select("evid", "e", "unit", "role").join(
            spark.createDataFrame(_pe[["evid", "direction"]].assign(headline=_pe["headline"].astype(int))), "evid", "inner")
        _hop = (payer_month.where(F.col("scope") == "int")
                .select(F.col("other").alias("unit"), F.col("e").alias("e2"), "n_risk", "n_broken", "n_grow")
                .join(F.broadcast(_u), "unit", "inner")
                .withColumn("t", F.col("e2") - F.col("e"))
                .where(F.col("t").between(-PRE, POST))
                .groupBy("direction", "headline", "role", "t").agg(
                    F.countDistinct("unit").alias("payers"), F.sum("n_risk").alias("streams"),
                    F.sum("n_broken").alias("broken"), F.sum("n_grow").alias("grown"))
                .toPandas())
    _h2rows, _bymo = [], []
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    for ax, d in zip(axes, DIRS):
        hd = _hop[_hop.direction == d]
        use = hd[hd.headline == 1]
        lab = HEAD_LABEL[d]
        if use[use.role == "S"].streams.sum() < 200:
            use, lab = hd, f"all {d} events in panel"
        g = use.groupby(["role", "t"])[["payers", "streams", "broken", "grown"]].sum().reset_index()
        if g.empty or g[g.role == "S"].streams.sum() == 0:
            _h2rows.append(dict(direction=d, group="no event customer pays other PNC customers through standing streams"))
            ax.axis("off")
            continue
        g["rate"] = (g.broken if d == "loss" else g.grown) / g.streams.where(g.streams > 0)
        for role, colr, nm in [("C", C_CTRL, "payees of matched controls"), ("S", C_DIR[d], f"payees of {d}-event customers")]:
            x = g[g.role == role].sort_values("t")
            ax.plot(x.t, 100 * x.rate, color=colr, marker="o", markersize=4, label=nm)
        ax.axvline(0, color=INK2, linewidth=1, linestyle=":")
        ax.set_title(f"{d.capitalize()}: own standing streams to PNC customers that {'break' if d == 'loss' else 'grow'}")
        ax.set_xlabel("months from the event"); ax.set_ylabel("% of standing streams"); ax.set_xticks(TS); ax.legend()

        def _rate(role, lo_t, hi_t):
            x = g[(g.role == role) & g.t.between(lo_t, hi_t)]
            return float((x.broken if d == "loss" else x.grown).sum() / max(x.streams.sum(), 1))
        sb, sa, cb, ca = _rate("S", -3, -1), _rate("S", 1, 3), _rate("C", -3, -1), _rate("C", 1, 3)
        _h2rows.append(dict(direction=d, group=lab, measure="break rate" if d == "loss" else "growth rate",
                            event_before=sb, event_after=sa, controls_before=cb, controls_after=ca,
                            extra_pp=100 * ((sa - sb) - (ca - cb)),
                            streams_event=int(g[g.role == "S"].streams.sum())))
        w = g.pivot(index="t", columns="role", values="rate").rename(columns={"S": f"{d}: event", "C": f"{d}: controls"})
        _bymo.append(w)
    fig.tight_layout()
    savefig(fig, "f14_second_hop", "6", "Second hop: own standing streams to other PNC customers that break (losses) or grow (gains)")
    _h2 = pd.DataFrame(_h2rows)
    _bym2 = pd.concat(_bymo, axis=1).reset_index() if _bymo else pd.DataFrame()
    CELL.show(("Second hop: months -3..-1 against +1..+3", _h2), ("By month", _bym2))
    for r in _h2.itertuples():
        if hasattr(r, "extra_pp") and np.isfinite(getattr(r, "extra_pp", np.nan)):
            ans(f"Second hop, {r.direction}: extra {r.measure} of own streams to PNC customers, +1..+3",
                f"{r.extra_pp:+.2f} pp ({r.group})")

## §7 · Case panels

The largest events among operating businesses (6 losses, 4 gains) plus named cases: monthly inflow as green
bars above zero, outflow as red bars below, the event month marked. The table lists the streams behind each event.

In [ ]:
if RUN_FULL:
    CELL.start("§7 · Case panels")
    with step("case selection and series"):
        _op = events[events.segment == "operating"].sort_values("amount", ascending=False)
        _cases = pd.concat([_op[_op.direction == "loss"].head(CONFIG["CASE_TOP_LOSS"]),
                            _op[_op.direction == "gain"].head(CONFIG["CASE_TOP_GAIN"])])
        _cases = _cases[["cust", "customer_name", "segment", "e", "direction", "amount", "size", "type", "breadth"]]
        for _p in CONFIG["CASE_NAME_PATTERNS"]:
            _nm = (shocks.where(F.upper(F.col("customer_name")).contains(_p.upper()))
                   .orderBy(F.desc("amount")).limit(1).toPandas())
            if len(_nm) and _nm.cust.iloc[0] not in set(_cases.cust):
                _cases = pd.concat([_cases, _nm[["cust", "customer_name", "segment", "e", "direction", "amount", "size",
                                                 "type", "breadth"]]], ignore_index=True)
        _ids = list(_cases.cust)
        _ser = pop.where(F.col("cust").isin(_ids)).select("cust", "mi", "amt_all_in", "amt_all_out").toPandas()
        _keys = spark.createDataFrame(_cases[["cust", "e"]].astype({"e": "int32"}))
        _br = (stream_events.join(F.broadcast(_keys), ["cust", "e"], "inner")
               .select("cust", "scope", "other", F.coalesce(F.col("ev"), F.lit("grow")).alias("event"), "expected",
                       (F.col("lost") + F.col("gain")).alias("dollars"), "pw_loss", "pw_gain", "n_risk", "n_broken", "n_grow")
               .unionByName(new_streams.join(F.broadcast(_keys), ["cust", "e"], "inner")
                            .select("cust", "scope", "other", F.lit("new payer").alias("event"),
                                    F.lit(None).cast("double").alias("expected"), F.col("new_amt").alias("dollars"),
                                    "pw_loss", "pw_gain", "n_risk", "n_broken", "n_grow"))
               .withColumn("rn", F.row_number().over(W.partitionBy("cust").orderBy(F.desc("dollars"))))
               .where(F.col("rn") <= 5).drop("rn").toPandas())
    for k, c in enumerate(_cases.itertuples(), start=1):
        s = _ser[_ser.cust == c.cust].set_index("mi").reindex(range(MAX_MI + 1)).fillna(0.0)
        fig, ax = plt.subplots(figsize=(11, 4))
        x = np.arange(MAX_MI + 1)
        ax.bar(x, s.amt_all_in / 1e6, color=C_IN, width=0.78, label="inflow")
        ax.bar(x, -s.amt_all_out / 1e6, color=C_OUT, width=0.78, label="outflow")
        ax.axhline(0, color=INK2, linewidth=1)
        ax.axvline(c.e, color=INK, linewidth=1.2, linestyle="--")
        ax.annotate(f"{c.direction} {mi2m(c.e)}: {usd(c.amount)} net ({c.size:.0%} of baseline), {c.type}",
                    xy=(c.e, ax.get_ylim()[1]), xytext=(4, -12), textcoords="offset points", fontsize=9, color=INK)
        ax.set_xticks(x[::2]); ax.set_xticklabels([mi2m(i) for i in x[::2]], rotation=60)
        ax.set_ylabel("USD millions per month"); ax.grid(axis="x", visible=False)
        ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, fontsize=9)
        ax.set_title(f"Case {k}: {c.customer_name}", pad=26)
        savefig(fig, f"f16_case_{k:02d}", "7", f"Case panel: {c.customer_name} ({c.direction}), inflow green, outflow red")
    _cases_show = _cases.assign(month=_cases.e.map(mi2m)).drop(columns=["e", "cust"])
    _br = _br.merge(_cases[["cust", "customer_name"]], on="cust").drop(columns=["cust"])
    CELL.show(("Cases", _cases_show[["customer_name", "segment", "month", "direction", "amount", "size", "type", "breadth"]]),
              ("Largest streams behind each case (other = payer id; n_risk / n_broken / n_grow = that payer's standing PNC payees)",
               _br[["customer_name", "scope", "other", "event", "expected", "dollars", "n_risk", "n_broken", "n_grow"]]))

## §8 · Answer sheet, gates and files

One output to screenshot at the end of a run.

In [ ]:
CELL.start("§8 · Answer sheet, gates and files")
pd.DataFrame(FIG_INDEX).to_csv(os.path.join(FIG_DIR, "figures_index.csv"), index=False)
gates_frame().to_csv(os.path.join(OUT_DIR, "p1_gates.csv"), index=False)
pd.DataFrame(ANSWERS).to_csv(os.path.join(OUT_DIR, "p1_answers.csv"), index=False)
CELL.show(("Answers", pd.DataFrame(ANSWERS)), ("Gates", gates_frame()),
          (f"Figures saved in {FIG_DIR}", pd.DataFrame(FIG_INDEX, columns=["file", "section", "caption"])),
          ("Timings (minutes)", pd.DataFrame(TIMINGS, columns=["step", "minutes"])))